In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.integrations.gguf.gguf_tokenizer_mapping import add_gguf_special_tokens


model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

Loading weights: 100%|██████████| 29/29 [00:00<00:00, 32732.73it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 2)
    (wpe): Embedding(1024, 2)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=6, nx=2)
          (c_proj): Conv1D(nf=2, nx=2)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=8, nx=2)
          (c_proj): Conv1D(nf=2, nx=8)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=2, out_features=50257, bias=False)
)

In [28]:
phrases = ["Hello world!",
           "The fucking mouth of this casino"]

for text in phrases:
    print(text)
    ids = tokenizer.encode(text, add_special_tokens=False)
    print(f"ID токенов: {ids}")
    print(f"Токены: {tokenizer.convert_ids_to_tokens(ids)}")

Hello world!
ID токенов: [15496, 995, 0]
Токены: ['Hello', 'Ġworld', '!']
The fucking mouth of this casino
ID токенов: [464, 9372, 5422, 286, 428, 21507]
Токены: ['The', 'Ġfucking', 'Ġmouth', 'Ġof', 'Ġthis', 'Ġcasino']


In [29]:
from transformers import GenerationConfig


sampling_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    num_beams=1,
    do_sample=True,
    max_new_tokens=50,
    top_k=50,
    temperature=0.8
)

greedy_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    num_beams=1,
    do_sample=False,
    max_new_tokens=50
)

promts = {
    "neutral" : "Large language model is",
    "instruction" : "Write what is large language model in one sentence.",
    "ambiguous": "I see"
}

In [30]:
for key, value in promts.items():
    for model_type, cfg in {"greedy":greedy_config, "sampling":sampling_config}.items():
        print(f"Model type: {model_type}")
        print(f"Prompt type: {key}\nPrompt: {value}")
        torch.manual_seed(42)
        tokens = tokenizer(value, return_tensors='pt', add_special_tokens=False)
        output = model.generate(**tokens, generation_config=cfg)
        print(f"Answer: {tokenizer.decode(output[0],skip_special_tokens=True)}\n")

Model type: greedy
Prompt type: neutral
Prompt: Large language model is
Answer: Large language model is factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors

Model type: sampling
Prompt type: neutral
Prompt: Large language model is
Answer: Large language model isozygGy incarcer448Mini praying Television Redux Singapore Boone omega Bend soy representations653 factors clearer brutality Pocket skillet Redux Bendacious courtyardMini Bend deflectived Singapore Tre courtyard TelevisionoblMostshowsived Medic skillet Television� Medic Singapore membership skillet courtyard factors clearerobl factors 236

Model type: greedy
Prompt type: instruction
Promp

In [31]:
for key, value in promts.items():
    for model_type, cfg in {"greedy":greedy_config, "sampling":sampling_config}.items():
        print(f"Model type: {model_type}")
        print(f"Prompt type: {key}\nPrompt: {value}")
        torch.manual_seed(52)
        tokens = tokenizer(value, return_tensors='pt', add_special_tokens=False)
        output = model.generate(**tokens, generation_config=cfg)
        print(f"Answer: {tokenizer.decode(output[0],skip_special_tokens=True)}\n")

Model type: greedy
Prompt type: neutral
Prompt: Large language model is
Answer: Large language model is factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors

Model type: sampling
Prompt type: neutral
Prompt: Large language model is
Answer: Large language model is factors BooneMost omega incarcer factors perhaps Singapore Boone factors Late deflectMiniMini ReduxSexual rubbing653showsMost boilsshowsozyg incarcer praying Boone linedProsSexual Television factors clearer skillet Dreams membership omega omega praying predators Dreams Pocket TreOutside incarcer Televisionozyg courtyard membershipPros�

Model type: greedy
Prompt type: instruction
Prompt